# Bias Correct DAMIP temperature data
 
> This notebook is part of the scientific analysis for the paper **"Heat-related mortality attributable to human-induced climate change: findings from the 100 Million Brazilian Cohort" (2025)**.  
>
> It contains code to process Detection and Attribution Model Intercomparison Project (DAMIP) model data and bias correct it to observations using the Inter-Sectorial Impact Model Intercomparison Project 3 (ISIMIP3) trend-preserving bias correction method.  
>  
> This notebook uses [code](https://zenodo.org/records/7151476) developed for the bias adjustment and statistical downscaling of ISIMIP3. 
>
> ---
>
> ### License
>
> This notebook is licensed under the [Creative Commons Attribution 4.0 International License (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/).  
You are free to share and adapt the material with appropriate credit.
>
>---
>
>### Authors
>
>- **Ruby Lieber** – School of Geographical Sciences, University of Bristol 
>- **Emily Vosper** – School of Geographical Sciences, University of Bristol 
>
>---
>
>### Dependencies
>
>This notebook was developed with **Python 3.13** and using **Jupyter Notebook**. Functions can be found in the GitHub repository. 
>
>Required packages (see also `pixi.toml`):  
>```bash
>numpy
>xarray
>xesmf
>```

#### Imports

In [1]:
import os
import re
import xarray as xr
import numpy as np
import xesmf as xe
from glob import glob
from collections import defaultdict
import functions as func
import subprocess
import tempfile

#### Observational data 
Daily average temperature was obtained from the Brazilian Daily Weather Gridded Data (BR-DWGD) produced by Xavier et al. (2022). NetCDF downloads are available [here](https://sites.google.com/site/alexandrecandidoxavierufes/brazilian-daily-weather-gridded-data?authuser=0). File paths will need to be changed. 

In [ ]:
# Read in obs data that is target for bias correction
obs = xr.open_dataarray('obs_padded_interp_01x01_2000_2018.nc')

#### DAMIP data 
Temperature (tas) data for three DAMIP models was downloaded from ESGF servers. File paths will need to be changed.

In [ ]:
# Directory containing files
base_path = "/processed_damip_data"

# Find all tas files
tas_files = glob(os.path.join(base_path, "tas_*.nc"))

# Pattern to extract model, run, and ensemble
pattern = re.compile(
    r"tas_(?P<model>[^_]+)_(?P<run>[^_]+)_(?P<ensemble>r\d+i\d+p\d+f\d+)_v\d+_(?P<yearrange>\d{4}-\d{4})\.nc"
)

# Group files by (model, run)
grouped_files = defaultdict(list)
for f in tas_files:
    match = pattern.search(os.path.basename(f))
    if match:
        key = (match["model"], match["run"])
        grouped_files[key].append((match["ensemble"], f))

# Read and combine files into datasets
datasets = {}
for (model, run), ens_files in grouped_files.items():
    # Sort by ensemble 
    ens_files.sort()
    
    # Load all files and assign ensemble coord
    members = []
    for ens_name, f in ens_files:
        ds = xr.open_dataset(f)
        ds = ds.expand_dims(ensemble=[ens_name])
        members.append(ds)

    combined = xr.concat(members, dim="ensemble")
    datasets[(model, run)] = combined

#### Preprocessing functions
These functions select out Brazil (our region of interest) and regrid the observations. Observations and model data need to be at the same spatial resolution prior to bias correction. Bias correction is performed at model resolution.

In [4]:
def process_model_data(
    data: xr.DataArray | xr.Dataset,
    region_from: xr.DataArray | xr.Dataset
) -> xr.DataArray:
    """
    Preprocess model data for bias correction.

    Steps:
    - Remove 'height' coordinate if present
    - Convert from Kelvin to Celsius
    - Fix longitudes
    - Regrid model data to a standard 1° or 2° grid depending on resolution
    - Select Brazil region based on 'region_from' bounding box
    - Standardize variable name, calendar, and chunking

    Parameters
    ----------
    data : xr.DataArray or xr.Dataset
        Raw model temperature data (e.g., tas).
    region_from : xr.DataArray or xr.Dataset
        Dataset used to define the Brazil bounding box (usually obs).

    Returns
    -------
    model_brazil : xr.DataArray
        Processed model data over Brazil, ready for bias correction.
    """
    data = data.copy()

    if "height" in data.coords:
        del data["height"]

    data = data - 273.15
    data = func.fix_longitudes(data)

    lat_step = 2 if float(np.diff(data.lat)[0]) > 1.5 else 1
    lon_step = 2 if float(np.diff(data.lon)[0]) > 1.5 else 1

    lats = np.arange(-90, 90.5, lat_step)
    lons = np.arange(-180, 180.5, lon_step)
    target_grid = xr.Dataset(coords={"lat": lats, "lon": lons})
    target_grid["dummy"] = (("lat", "lon"), np.zeros((len(lats), len(lons))))

    data.data = np.ascontiguousarray(data.data)

    regridder = xe.Regridder(data, target_grid, method="bilinear", unmapped_to_nan=True, reuse_weights=False)
    model_rg = regridder(data)
    model_rg = model_rg.chunk({'time': -1, 'lat': 60, 'lon': 60})

    lat_min = float(region_from.lat.min())
    lat_max = float(region_from.lat.max())
    lon_min = float(region_from.lon.min())
    lon_max = float(region_from.lon.max())

    lat_tol = lat_step / 2
    lon_tol = lon_step / 2

    lat_slice = slice(
        model_rg.lat.sel(lat=lat_min - lat_tol, method="nearest").values,
        model_rg.lat.sel(lat=lat_max + lat_tol, method="nearest").values
    )
    lon_slice = slice(
        model_rg.lon.sel(lon=lon_min - lon_tol, method="nearest").values,
        model_rg.lon.sel(lon=lon_max + lon_tol, method="nearest").values
    )

    model_brazil = model_rg.sel(lat=lat_slice, lon=lon_slice)
    model_brazil.name = "tas"
    model_brazil = model_brazil.transpose("lat", "lon", "time").chunk("auto")
    model_brazil = model_brazil.convert_calendar("proleptic_gregorian", use_cftime=True)
    model_brazil['time'] = model_brazil['time'].dt.floor('D')

    return model_brazil

In [5]:
def regrid_obs_to_model(
    obs: xr.DataArray | xr.Dataset,
    model_brazil: xr.DataArray
) -> xr.DataArray:
    """
    Regrid observation data to match the processed model grid over Brazil.

    Parameters
    ----------
    obs : xr.DataArray or xr.Dataset
        Raw global observation dataset.
    model_brazil : xr.DataArray
        Preprocessed model data defining the target grid over Brazil.

    Returns
    -------
    obs_rg : xr.DataArray
        Observation data regridded to match the model grid.
    """
    obs = obs.copy()

    regridder = xe.Regridder(obs, model_brazil, method="bilinear", unmapped_to_nan=True, reuse_weights=False)
    obs_rg = regridder(obs)

    obs_rg.name = "tas"
    obs_rg = obs_rg.transpose("lat", "lon", "time").chunk("auto")
    obs_rg = obs_rg.convert_calendar("proleptic_gregorian", use_cftime=True)

    return obs_rg

#### Apply bias correction
Bias correction is trained on the historical model run and then applied to the hist-nat model run. The script "bias_adjustment.py" is run using a subprocess within the jupyter notebook. It can be run through the command line if preprocessed files are saved.

In [ ]:
# Select model if wanting to do one at a time
# datasets = {key: datasets[key] for key in [('CanESM5', 'historical'), ('CanESM5', 'hist-nat')]}
# datasets = {key: datasets[key] for key in [('MRI-ESM2-0', 'historical'), ('MRI-ESM2-0', 'hist-nat')]}
# datasets = {key: datasets[key] for key in [('GFDL-CM4', 'historical'), ('GFDL-CM4', 'hist-nat')]}

In [ ]:
# Bias correct hist-nat 
for (model, run) in datasets:
    if run != "historical":
        continue

    try:
        training_data_ds = datasets[(model, "historical")] 
        experiment_data_ds = datasets[(model, "hist-nat")] # Apply bias correction to hist-nat run 
    except KeyError:
        print(f"Skipping {model}: hist-nat run not found")
        continue

    for ens in training_data_ds.ensemble.values:
        if ens not in experiment_data_ds.ensemble:
            print(f"Skipping {model} {ens}: not in hist-nat")
            continue

        training_data = training_data_ds.sel(ensemble=ens).tas
        experiment_data = experiment_data_ds.sel(ensemble=ens).tas

        # Process data
        training_processed = process_model_data(training_data, obs)
        experiment_processed = process_model_data(experiment_data, obs)
        obs_processed = regrid_obs_to_model(obs, training_processed)

        # Drop ensemble coord
        experiment_processed = experiment_processed.reset_coords('ensemble', drop=True)

        with tempfile.TemporaryDirectory() as tmpdir:
            obs_path = f"{tmpdir}/tas_obs.nc"
            train_path = f"{tmpdir}/tas_train.nc"
            model_path = f"{tmpdir}/tas_model.nc"
            bias_path = f"/bias_correct_damip/tas_{model}_hist-nat_{ens}_bias_corrected_2000_2018.nc"

            # Save inputs
            obs_processed.to_netcdf(obs_path)
            training_processed.to_netcdf(train_path)
            experiment_processed.to_netcdf(model_path)

            # Run bias correction script
            cmd = [
                "python", "-u", "bias_adjustment.py",
                "--n-processes", "12",
                "--randomization-seed", "0",
                "--step-size", "1",
                "-v", "tas",
                "--distribution", "normal",
                "-t", "additive",
                "-d", "1",
                "-w", "0",
                "-o", obs_path,
                "-s", train_path,
                "-f", model_path,
                "-b", bias_path
            ]

            print(f"Running bias correction for {model} {ens}...")
            result = subprocess.run(cmd, capture_output=True, text=True)
            # print(result.stdout)
            if result.returncode != 0:
                print(f"Error in {model} {ens}:", result.stderr)

In [ ]:
# Bias correct historical
for (model, run) in datasets:
    if run != "historical":
        continue

    try:
        training_data_ds = datasets[(model, "historical")]
        experiment_data_ds = datasets[(model, "historical")] # Apply to historical runs this time 
    except KeyError:
        print(f"Skipping {model}: hist-nat run not found")
        continue

    for ens in training_data_ds.ensemble.values:
        if ens not in experiment_data_ds.ensemble:
            print(f"Skipping {model} {ens}: not in hist-nat")
            continue

        training_data = training_data_ds.sel(ensemble=ens).tas
        experiment_data = experiment_data_ds.sel(ensemble=ens).tas

        # Process data
        training_processed = process_model_data(training_data, obs)
        experiment_processed = process_model_data(experiment_data, obs)
        obs_processed = regrid_obs_to_model(obs, training_processed)

        # Drop ensemble coord
        experiment_processed = experiment_processed.reset_coords('ensemble', drop=True)

        with tempfile.TemporaryDirectory() as tmpdir:
            obs_path = f"{tmpdir}/tas_obs.nc"
            train_path = f"{tmpdir}/tas_train.nc"
            model_path = f"{tmpdir}/tas_model.nc"
            bias_path = f"/bias_correct_damip/tas_{model}_historical_{ens}_bias_corrected_2000_2018.nc"

            # Save inputs
            obs_processed.to_netcdf(obs_path)
            training_processed.to_netcdf(train_path)
            experiment_processed.to_netcdf(model_path)

            # Run bias correction script
            cmd = [
                "python", "-u", "bias_adjustment.py",
                "--n-processes", "12",
                "--randomization-seed", "0",
                "--step-size", "1",
                "-v", "tas",
                "--distribution", "normal",
                "-t", "additive",
                "-d", "1",
                "-w", "0",
                "-o", obs_path,
                "-s", train_path,
                "-f", model_path,
                "-b", bias_path
            ]

            print(f"Running bias correction for {model} {ens}...")
            result = subprocess.run(cmd, capture_output=True, text=True)
            # print(result.stdout)
            if result.returncode != 0:
                print(f"Error in {model} {ens}:", result.stderr)